In [ ]:
# ============================================================
# MEAL PLANNER AI MODEL - FULL GOOGLE COLAB SCRIPT
# ============================================================

# 1. Install / import library
import pandas as pd
import joblib
import json
import random
from google.colab import files

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries loaded")


# ============================================================
# 2. Upload meals.csv
# ============================================================

print("📁 Upload file meals.csv kamu sekarang...")
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded file: {csv_filename}")


# ============================================================
# 3. Load dataset
# ============================================================

df = pd.read_csv(csv_filename)
df = df.fillna("")

print("✅ Dataset loaded")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

display(df.head())


# ============================================================
# 4. Validate required columns
# ============================================================

required_columns = [
    "id",
    "name",
    "meal_type",
    "calories",
    "protein",
    "carbs",
    "fat",
    "ingredients",
    "diet_tags",
    "allergen_tags",
    "budget_level",
    "cooking_time",
    "cuisine",
    "difficulty"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"❌ Missing columns: {missing_columns}")

print("✅ All required columns exist")


# ============================================================
# 5. Clean numeric columns
# ============================================================

numeric_columns = ["calories", "protein", "carbs", "fat", "cooking_time"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("✅ Numeric columns cleaned")


# ============================================================
# 6. Create features column for AI recommender
# ============================================================

df["features"] = (
    df["meal_type"].astype(str) + " " +
    df["ingredients"].astype(str) + " " +
    df["diet_tags"].astype(str) + " " +
    df["budget_level"].astype(str) + " " +
    df["cuisine"].astype(str) + " " +
    df["difficulty"].astype(str)
)

print("✅ Features column created")
display(df[["name", "features"]].head())


# ============================================================
# 7. Train TF-IDF recommender model
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

meal_vectors = vectorizer.fit_transform(df["features"])

print("✅ Recommender model trained")
print("Meal vectors shape:", meal_vectors.shape)


# ============================================================
# 8. Rules Engine: filter meal by allergy, diet, budget, time
# ============================================================

def filter_meals(
    data,
    allergies=None,
    diet=None,
    max_cooking_time=None,
    budget=None,
    meal_type=None,
    cuisine=None
):
    filtered = data.copy()
    allergies = allergies or []

    # Allergy filter
    for allergy in allergies:
        allergy = allergy.strip().lower()
        if allergy:
            filtered = filtered[
                ~filtered["allergen_tags"].str.lower().str.contains(allergy, na=False)
            ]

    # Diet filter
    if diet:
        diet = diet.strip().lower()
        filtered = filtered[
            filtered["diet_tags"].str.lower().str.contains(diet, na=False)
        ]

    # Max cooking time filter
    if max_cooking_time:
        filtered = filtered[
            filtered["cooking_time"] <= max_cooking_time
        ]

    # Budget filter
    if budget:
        budget = budget.strip().lower()
        filtered = filtered[
            filtered["budget_level"].str.lower() == budget
        ]

    # Meal type filter
    if meal_type:
        meal_type = meal_type.strip().lower()
        filtered = filtered[
            filtered["meal_type"].str.lower() == meal_type
        ]

    # Cuisine filter, soft filter.
    # Kalau hasil cuisine terlalu sedikit, jangan dipaksa.
    if cuisine:
        cuisine = cuisine.strip().lower()
        cuisine_filtered = filtered[
            filtered["cuisine"].str.lower().str.contains(cuisine, na=False)
        ]

        if len(cuisine_filtered) >= 1:
            filtered = cuisine_filtered

    return filtered


# ============================================================
# 9. Recommendation function
# ============================================================

def recommend_meals(user_profile, top_n=5):
    query = " ".join([
        str(user_profile.get("meal_type", "")),
        str(user_profile.get("diet", "")),
        str(user_profile.get("budget", "")),
        str(user_profile.get("cuisine", "")),
        str(user_profile.get("goal", "")),
        str(user_profile.get("difficulty", ""))
    ])

    filtered_df = filter_meals(
        df,
        allergies=user_profile.get("allergies", []),
        diet=user_profile.get("diet"),
        max_cooking_time=user_profile.get("max_cooking_time"),
        budget=user_profile.get("budget"),
        meal_type=user_profile.get("meal_type"),
        cuisine=user_profile.get("cuisine")
    )

    if filtered_df.empty:
        return pd.DataFrame()

    filtered_vectors = vectorizer.transform(filtered_df["features"])
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(query_vector, filtered_vectors).flatten()

    result = filtered_df.copy()
    result["score"] = scores

    result = result.sort_values("score", ascending=False).head(top_n)

    return result


print("✅ Recommendation function ready")


# ============================================================
# 10. Select one meal with controlled randomness
# ============================================================

used_meals = set()

def pick_meal(user_profile, meal_type, top_n=8):
    global used_meals

    profile = user_profile.copy()
    profile["meal_type"] = meal_type

    recommendations = recommend_meals(profile, top_n=top_n)

    if recommendations.empty:
        return None

    # Avoid repeated meals if possible
    fresh_recommendations = recommendations[
        ~recommendations["name"].isin(used_meals)
    ]

    if fresh_recommendations.empty:
        fresh_recommendations = recommendations

    selected = fresh_recommendations.sample(1).iloc[0]
    used_meals.add(selected["name"])

    return {
        "id": selected["id"],
        "name": selected["name"],
        "meal_type": selected["meal_type"],
        "calories": int(selected["calories"]),
        "protein": int(selected["protein"]),
        "carbs": int(selected["carbs"]),
        "fat": int(selected["fat"]),
        "ingredients": selected["ingredients"],
        "diet_tags": selected["diet_tags"],
        "allergen_tags": selected["allergen_tags"],
        "budget_level": selected["budget_level"],
        "cooking_time": int(selected["cooking_time"]),
        "cuisine": selected["cuisine"],
        "difficulty": selected["difficulty"],
        "score": round(float(selected["score"]), 4)
    }


# ============================================================
# 11. Generate daily meal plan
# ============================================================

def generate_day_plan(user_profile, include_snack=True):
    meal_types = ["breakfast", "lunch", "dinner"]

    if include_snack:
        meal_types.append("snack")

    day_plan = {
        "meals": {},
        "daily_total": {
            "calories": 0,
            "protein": 0,
            "carbs": 0,
            "fat": 0
        }
    }

    for meal_type in meal_types:
        meal = pick_meal(user_profile, meal_type)

        day_plan["meals"][meal_type] = meal

        if meal:
            day_plan["daily_total"]["calories"] += meal["calories"]
            day_plan["daily_total"]["protein"] += meal["protein"]
            day_plan["daily_total"]["carbs"] += meal["carbs"]
            day_plan["daily_total"]["fat"] += meal["fat"]

    return day_plan


# ============================================================
# 12. Generate 7-day meal plan
# ============================================================

def generate_week_plan(user_profile, include_snack=True):
    global used_meals
    used_meals = set()

    days = [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]

    week_plan = {
        "user_profile": user_profile,
        "days": []
    }

    for day in days:
        day_plan = generate_day_plan(user_profile, include_snack=include_snack)
        day_plan["day"] = day
        week_plan["days"].append(day_plan)

    return week_plan


print("✅ Meal plan generator ready")


# ============================================================
# 13. Generate shopping list
# ============================================================

def generate_shopping_list(week_plan):
    ingredients = []

    for day in week_plan["days"]:
        for meal_type, meal in day["meals"].items():
            if meal and meal.get("ingredients"):
                meal_ingredients = [
                    item.strip().lower()
                    for item in str(meal["ingredients"]).split(",")
                    if item.strip()
                ]
                ingredients.extend(meal_ingredients)

    shopping_counts = {}

    for item in ingredients:
        shopping_counts[item] = shopping_counts.get(item, 0) + 1

    shopping_list = [
        {
            "ingredient": item,
            "frequency": count
        }
        for item, count in sorted(
            shopping_counts.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    return shopping_list


# ============================================================
# 14. Test user profile
# Kamu bisa edit bagian ini sesuai kebutuhan
# ============================================================

user_profile = {
    "diet": "halal",
    "budget": "low",
    "cuisine": "indonesian",
    "goal": "high_protein balanced",
    "difficulty": "easy",
    "allergies": ["peanut"],
    "max_cooking_time": 30
}

print("🧪 Testing with user profile:")
print(json.dumps(user_profile, indent=2))


# ============================================================
# 15. Test recommendation
# ============================================================

test_profile = user_profile.copy()
test_profile["meal_type"] = "dinner"

test_result = recommend_meals(test_profile, top_n=5)

print("✅ Top dinner recommendations:")
display(test_result[
    [
        "name",
        "meal_type",
        "calories",
        "protein",
        "ingredients",
        "allergen_tags",
        "budget_level",
        "cooking_time",
        "score"
    ]
])


# ============================================================
# 16. Generate week plan
# ============================================================

week_plan = generate_week_plan(user_profile, include_snack=True)
week_plan["shopping_list"] = generate_shopping_list(week_plan)

print("✅ 7-day meal plan generated")


# ============================================================
# 17. Display week plan summary
# ============================================================

for day in week_plan["days"]:
    print("\n==============================")
    print(day["day"])
    print("==============================")

    for meal_type, meal in day["meals"].items():
        if meal:
            print(f"{meal_type.title()}: {meal['name']} | {meal['calories']} kcal | {meal['protein']}g protein")
        else:
            print(f"{meal_type.title()}: No meal found")

    total = day["daily_total"]
    print(
        f"Total: {total['calories']} kcal | "
        f"{total['protein']}g protein | "
        f"{total['carbs']}g carbs | "
        f"{total['fat']}g fat"
    )


# ============================================================
# 18. Save model and output files
# ============================================================

joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(df, "meals_database.pkl")

with open("week_plan.json", "w", encoding="utf-8") as f:
    json.dump(week_plan, f, indent=2, ensure_ascii=False)

with open("shopping_list.json", "w", encoding="utf-8") as f:
    json.dump(week_plan["shopping_list"], f, indent=2, ensure_ascii=False)

print("✅ Files saved:")
print("- vectorizer.pkl")
print("- meals_database.pkl")
print("- week_plan.json")
print("- shopping_list.json")


# ============================================================
# 19. Download files
# ============================================================

files.download("vectorizer.pkl")
files.download("meals_database.pkl")
files.download("week_plan.json")
files.download("shopping_list.json")

print("🎉 Done. Meal Planner AI model is ready.")